In [11]:
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

import time
import warnings

import joblib
import numpy as np
import polars as pl
import tensorflow as tf
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

warnings.filterwarnings('ignore', message='X does not have valid feature names')
warnings.filterwarnings('ignore', category=UserWarning, module='xgboost')

SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

TARGET_COL = 'label'
N_REPEATS = 3
WARMUP_SIZE = 1000


In [12]:
# Carga de dataset y preprocesado, igual que en los cuadernos de ataques para IOT-23
prepared_path = '../../DATASETS/dataSets_Reducidos/iot-23/datos_IOT_23_preparado.csv'

df_encoded = pl.read_csv(prepared_path)

feature_columns = [col for col in df_encoded.columns if col != TARGET_COL]
X = df_encoded.select(feature_columns)
y_np = df_encoded[TARGET_COL].to_numpy().astype(np.int8)
X_np = X.to_numpy().astype(np.float32)

indices = np.arange(X_np.shape[0])

train_full_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=SEED,
    stratify=y_np,
)

X_full_train_np = X_np[train_full_idx]
X_test_np = X_np[test_idx]

X_full_base = X_np.astype(np.float32)

mlp_scaler = StandardScaler()
mlp_scaler.fit(X_full_train_np)
X_full_scaled_mlp = mlp_scaler.transform(X_full_base).astype(np.float32)

cnn_scaler = MinMaxScaler()
cnn_scaler.fit(X_full_train_np)
X_full_scaled_cnn = cnn_scaler.transform(X_full_base).astype(np.float32)
X_full_cnn = X_full_scaled_cnn.reshape(X_full_scaled_cnn.shape[0], X_full_scaled_cnn.shape[1], 1)

N_MUESTRAS = len(X_full_base)

print(f'Train full: {len(X_full_train_np):,} muestras')
print(f'Test:       {len(X_test_np):,} muestras')
print(f'Total:      {N_MUESTRAS:,} muestras')
print(f'Features:   {X_full_base.shape[1]}')


Train full: 931,880 muestras
Test:       232,971 muestras
Total:      1,164,851 muestras
Features:   22


In [13]:
# RF

rf_path = '../model/iot-23/rf_iot23.joblib'

rf_model = joblib.load(rf_path)
print('Modelo cargado')

# Warm-up
_ = rf_model.predict(X_full_base[:min(WARMUP_SIZE, len(X_full_base))])

# Medida latencia
tiempos_rf = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = rf_model.predict(X_full_base)
    t1 = time.perf_counter()
    tiempos_rf.append(t1 - t0)

tiempo_total_rf = float(np.mean(tiempos_rf))
throughput_rf = N_MUESTRAS / tiempo_total_rf

print(f'Tiempos medidos: {[round(t, 4) for t in tiempos_rf]}')
print(f'Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_rf:.4f} s')
print(f'Throughput aproximado: {throughput_rf:,.0f} muestras/s')


Modelo cargado
Tiempos medidos: [1.009, 1.0451, 0.9852]
Tiempo medio de inferencia sobre todo el dataset: 1.0131 s
Throughput aproximado: 1,149,778 muestras/s


In [14]:
# XGBOOST

xgb_path = '../model/iot-23/xgb_iot23.joblib'

xgb_model = joblib.load(xgb_path)
print('Modelo cargado')

# Warm-up
_ = xgb_model.predict(X_full_base[:min(WARMUP_SIZE, len(X_full_base))])

# Medida latencia
tiempos_xgb = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = xgb_model.predict(X_full_base)
    t1 = time.perf_counter()
    tiempos_xgb.append(t1 - t0)

tiempo_total_xgb = float(np.mean(tiempos_xgb))
throughput_xgb = N_MUESTRAS / tiempo_total_xgb

print(f'Tiempos medidos: {[round(t, 4) for t in tiempos_xgb]}')
print(f'Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_xgb:.4f} s')
print(f'Throughput aproximado: {throughput_xgb:,.0f} muestras/s')


Modelo cargado


/usr/lib/python3.12/pickle.py:1710: UserWarning: [11:06:39] WARNING: /__w/xgboost/xgboost/src/gbm/gbtree.cc:402: Changing updater from `grow_gpu_hist` to `grow_quantile_histmaker`.
  setstate(state)
/usr/lib/python3.12/pickle.py:1710: UserWarning: [11:06:39] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  setstate(state)
/usr/lib/python3.12/pickle.py:1710: UserWarning: [11:06:39] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  setstate(state)


Tiempos medidos: [0.2522, 0.2266, 0.1693]
Tiempo medio de inferencia sobre todo el dataset: 0.2160 s
Throughput aproximado: 5,391,713 muestras/s


In [15]:
# LIGHTGBM

lgbm_path = '../model/iot-23/lgbm_iot23.joblib'

lgbm_model = joblib.load(lgbm_path)
print('Modelo cargado')

# Warm-up
_ = lgbm_model.predict(X_full_base[:min(WARMUP_SIZE, len(X_full_base))])

# Medida latencia
tiempos_lgbm = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = lgbm_model.predict(X_full_base)
    t1 = time.perf_counter()
    tiempos_lgbm.append(t1 - t0)

tiempo_total_lgbm = float(np.mean(tiempos_lgbm))
throughput_lgbm = N_MUESTRAS / tiempo_total_lgbm

print(f'Tiempos medidos: {[round(t, 4) for t in tiempos_lgbm]}')
print(f'Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_lgbm:.4f} s')
print(f'Throughput aproximado: {throughput_lgbm:,.0f} muestras/s')


Modelo cargado
Tiempos medidos: [0.2547, 0.3154, 0.3186]
Tiempo medio de inferencia sobre todo el dataset: 0.2962 s
Throughput aproximado: 3,932,579 muestras/s


In [16]:
# CATBOOST

catboost_path = '../model/iot-23/catboost_iot23.joblib'

catboost_model = joblib.load(catboost_path)
print('Modelo cargado')

# Warm-up
_ = catboost_model.predict(X_full_base[:min(WARMUP_SIZE, len(X_full_base))])

# Medida latencia
tiempos_catboost = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = catboost_model.predict(X_full_base)
    t1 = time.perf_counter()
    tiempos_catboost.append(t1 - t0)

tiempo_total_catboost = float(np.mean(tiempos_catboost))
throughput_catboost = N_MUESTRAS / tiempo_total_catboost

print(f'Tiempos medidos: {[round(t, 4) for t in tiempos_catboost]}')
print(f'Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_catboost:.4f} s')
print(f'Throughput aproximado: {throughput_catboost:,.0f} muestras/s')


Modelo cargado
Tiempos medidos: [0.3093, 0.2942, 0.302]
Tiempo medio de inferencia sobre todo el dataset: 0.3018 s
Throughput aproximado: 3,859,140 muestras/s


In [17]:
# SVM

svm_path = '../model/iot-23/svm_iot23.joblib'

svm_model = joblib.load(svm_path)
print('Modelo cargado')

# Warm-up
_ = svm_model.predict(X_full_base[:min(WARMUP_SIZE, len(X_full_base))])

# Medida latencia
tiempos_svm = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = svm_model.predict(X_full_base)
    t1 = time.perf_counter()
    tiempos_svm.append(t1 - t0)

tiempo_total_svm = float(np.mean(tiempos_svm))
throughput_svm = N_MUESTRAS / tiempo_total_svm

print(f'Tiempos medidos: {[round(t, 4) for t in tiempos_svm]}')
print(f'Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_svm:.4f} s')
print(f'Throughput aproximado: {throughput_svm:,.0f} muestras/s')


Modelo cargado
Tiempos medidos: [0.2497, 0.196, 0.2108]
Tiempo medio de inferencia sobre todo el dataset: 0.2188 s
Throughput aproximado: 5,323,551 muestras/s


In [18]:
# MLP

mlp_path = '../model/iot-23/mlp_iot23.joblib'

mlp_model = joblib.load(mlp_path)
print('Modelo cargado')

# Warm-up
_ = mlp_model.predict(X_full_scaled_mlp[:min(WARMUP_SIZE, len(X_full_scaled_mlp))], batch_size=4096, verbose=0)

# Medida latencia
tiempos_mlp = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = mlp_model.predict(X_full_scaled_mlp, batch_size=4096, verbose=0)
    t1 = time.perf_counter()
    tiempos_mlp.append(t1 - t0)

tiempo_total_mlp = float(np.mean(tiempos_mlp))
throughput_mlp = N_MUESTRAS / tiempo_total_mlp

print(f'Tiempos medidos: {[round(t, 4) for t in tiempos_mlp]}')
print(f'Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_mlp:.4f} s')
print(f'Throughput aproximado: {throughput_mlp:,.0f} muestras/s')


Modelo cargado
Tiempos medidos: [0.7171, 0.5626, 0.5023]
Tiempo medio de inferencia sobre todo el dataset: 0.5940 s
Throughput aproximado: 1,961,042 muestras/s


In [19]:
# CNN

cnn_path = '../model/iot-23/cnn_iot23.joblib'

cnn_model = joblib.load(cnn_path)
print('Modelo cargado')

# Warm-up
_ = cnn_model.predict(X_full_cnn[:min(WARMUP_SIZE, len(X_full_cnn))], batch_size=4096, verbose=0)

# Medida latencia
tiempos_cnn = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = cnn_model.predict(X_full_cnn, batch_size=4096, verbose=0)
    t1 = time.perf_counter()
    tiempos_cnn.append(t1 - t0)

tiempo_total_cnn = float(np.mean(tiempos_cnn))
throughput_cnn = N_MUESTRAS / tiempo_total_cnn

print(f'Tiempos medidos: {[round(t, 4) for t in tiempos_cnn]}')
print(f'Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_cnn:.4f} s')
print(f'Throughput aproximado: {throughput_cnn:,.0f} muestras/s')


Modelo cargado
Tiempos medidos: [6.5198, 6.21, 5.9515]
Tiempo medio de inferencia sobre todo el dataset: 6.2271 s
Throughput aproximado: 187,062 muestras/s


In [20]:
# Tabla comparativa final

tabla_comparativa = pd.DataFrame(
    [
        {'Modelo': 'RF', 'Tiempo medio (s)': tiempo_total_rf, 'Muestras/s aprox': throughput_rf},
        {'Modelo': 'XGBoost', 'Tiempo medio (s)': tiempo_total_xgb, 'Muestras/s aprox': throughput_xgb},
        {'Modelo': 'LightGBM', 'Tiempo medio (s)': tiempo_total_lgbm, 'Muestras/s aprox': throughput_lgbm},
        {'Modelo': 'CatBoost', 'Tiempo medio (s)': tiempo_total_catboost, 'Muestras/s aprox': throughput_catboost},
        {'Modelo': 'SVM', 'Tiempo medio (s)': tiempo_total_svm, 'Muestras/s aprox': throughput_svm},
        {'Modelo': 'MLP', 'Tiempo medio (s)': tiempo_total_mlp, 'Muestras/s aprox': throughput_mlp},
        {'Modelo': 'CNN', 'Tiempo medio (s)': tiempo_total_cnn, 'Muestras/s aprox': throughput_cnn},
    ]
).sort_values('Tiempo medio (s)').reset_index(drop=True)

tabla_comparativa['Tiempo medio (s)'] = tabla_comparativa['Tiempo medio (s)'].round(4)
tabla_comparativa['Muestras/s aprox'] = tabla_comparativa['Muestras/s aprox'].round(0).astype(int)

display(tabla_comparativa)


,Modelo,Tiempo medio (s),Muestras/s aprox
0,XGBoost,0.2160,5391713
1,SVM,0.2188,5323551
2,LightGBM,0.2962,3932579
3,CatBoost,0.3018,3859140
4,MLP,0.5940,1961042
5,RF,1.0131,1149778
6,CNN,6.2271,187062
